In [ ]:
import os
import shutil
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import urllib.parse

load_dotenv()

SRC_USER = os.getenv("DB_USER")
SRC_PASSWORD = os.getenv("DB_PASSWORD")
SRC_HOST = os.getenv("DB_HOST")
SRC_PORT = os.getenv("DB_PORT")
SRC_NAME = os.getenv("DB_NAME")

src_safe_password = urllib.parse.quote_plus(SRC_PASSWORD)
src_url = f"mysql+pymysql://{SRC_USER}:{src_safe_password}@{SRC_HOST}:{SRC_PORT}/{SRC_NAME}"
src_engine = create_engine(src_url)

TARGET_DB_USER = "****"
TARGET_DB_PASSWORD = "****"
TARGET_DB_HOST = "****"
TARGET_DB_PORT = "****"
TARGET_DB_NAME = "****"

new_safe_password = urllib.parse.quote_plus(TARGET_DB_PASSWORD)
target_db_url = f"mysql+pymysql://{TARGET_DB_USER}:{new_safe_password}@{TARGET_DB_HOST}:{TARGET_DB_PORT}/{TARGET_DB_NAME}"
target_engine = create_engine(target_db_url)


In [ ]:
query_lotids = """
    SELECT DISTINCT a.LOTID
    FROM ai_proc_davalue a
    INNER JOIN ai_proc_prevalue b ON a.LOTID = b.LOTID
    WHERE a.isvalid = 'O'
"""
df_lotids = pd.read_sql(query_lotids, src_engine)
print(f"▶ 양쪽 테이블에 모두 존재하는 유효 고유 LOTID 수: {len(df_lotids)}개")

if len(df_lotids) < 2000:
    raise ValueError(f"조건을 만족하는 고유 LOTID가 {len(df_lotids)}개로 부족합니다.")

# 2. 2,000개 랜덤 샘플링
sampled_lotids = df_lotids['LOTID'].sample(n=2000, random_state=42).tolist()
lotid_tuple = tuple(sampled_lotids)
print("▶ 2,000개 LOTID 샘플링 완료. 상세 데이터를 추출합니다...")

# 3. 샘플링된 LOTID에 대한 davalue(원천) 데이터 추출
query_raw = f"SELECT * FROM ai_proc_davalue WHERE LOTID IN {lotid_tuple}"
df_raw = pd.read_sql(query_raw, src_engine)
print(f"▶ 원천 데이터(davalue) 로드 완료: {len(df_raw)}건")

# 4. 샘플링된 LOTID에 대한 prevalue(전처리) 데이터 추출
query_pre = f"SELECT * FROM ai_proc_prevalue WHERE LOTID IN {lotid_tuple}"
df_pre = pd.read_sql(query_pre, src_engine)
print(f"▶ 전처리 데이터(prevalue) 로드 완료: {len(df_pre)}건")

In [ ]:
# 테이블명 정의
raw_table = 'wm_raw_unstructed_datasets'
pre_table = 'wm_unstructed_datasets'

print("▶ Target DB에 데이터 적재 및 PK 설정을 시작합니다...")

with target_engine.begin() as connection:
    # 1. 원천 데이터(raw) 적재 및 PK 설정
    df_raw.to_sql(name=raw_table, con=target_engine, if_exists='replace', index=False)
    
    connection.execute(text(f"ALTER TABLE {raw_table} MODIFY LOTID VARCHAR(50);"))
    connection.execute(text(f"ALTER TABLE {raw_table} ADD COLUMN SEQ_ID INT AUTO_INCREMENT PRIMARY KEY FIRST;"))
    print(f"  - {raw_table} 적재 및 PK 설정 완료.")

    # 2. 전처리 데이터(pre) 적재 및 PK 설정
    df_pre.to_sql(name=pre_table, con=target_engine, if_exists='replace', index=False)
    
    connection.execute(text(f"ALTER TABLE {pre_table} MODIFY LOTID VARCHAR(50);"))
    connection.execute(text(f"ALTER TABLE {pre_table} ADD COLUMN SEQ_ID INT AUTO_INCREMENT PRIMARY KEY FIRST;"))
    print(f"  - {pre_table} 적재 및 PK 설정 완료.")

print("▶ [최종 완료] Step 1~3 데이터 마이그레이션이 성공적으로 끝났습니다.")

In [ ]:
# 1. 대상 DB의 두 테이블에서 각각 고유 LOTID만 추출
query_raw_lot = "SELECT DISTINCT LOTID FROM wm_raw_unstructed_datasets"
query_pre_lot = "SELECT DISTINCT LOTID FROM wm_unstructed_datasets"

print("▶ Target DB에서 고유 LOTID 개수 확인 중...")
df_raw_lot = pd.read_sql(query_raw_lot, target_engine)
df_pre_lot = pd.read_sql(query_pre_lot, target_engine)

raw_count = len(df_raw_lot)
pre_count = len(df_pre_lot)

print(f"  - wm_raw_unstructed_datasets 고유 LOTID 수 : {raw_count}개")
print(f"  - wm_unstructed_datasets 고유 LOTID 수     : {pre_count}개")

# 2. 파이썬 Set(집합)을 이용해 두 LOTID 그룹이 완전히 일치하는지 검증
set_raw = set(df_raw_lot['LOTID'])
set_pre = set(df_pre_lot['LOTID'])

if set_raw == set_pre:
    print("\n▶ [검증 성공] 두 테이블의 고유 LOTID가 100% 완벽하게 일치합니다!")
else:
    print("\n▶ [검증 실패] 두 테이블의 LOTID 구성이 다릅니다.")
    
    # 어디서 누락/차이가 발생했는지 상세 분석 (대칭 차집합)
    diff_raw_only = set_raw - set_pre
    diff_pre_only = set_pre - set_raw
    
    if diff_raw_only:
        print(f"  - 원천(raw) 테이블에만 있는 LOTID ({len(diff_raw_only)}개): {list(diff_raw_only)[:5]} ...")
    if diff_pre_only:
        print(f"  - 전처리(pre) 테이블에만 있는 LOTID ({len(diff_pre_only)}개): {list(diff_pre_only)[:5]} ...")

In [ ]:
# 1. Source DB에서 2,000개 LOTID에 해당하는 원본 이미지 경로(ai_vision_davalue) 조회
query_vision = f"SELECT DISTINCT LOTID, FILEPATH AS old_filepath FROM ai_vision_davalue WHERE LOTID IN {lotid_tuple}"
df_vision = pd.read_sql(query_vision, src_engine)

print(f"▶ 원본 이미지 경로(ai_vision_davalue) 조회 완료: {len(df_vision)}건")

# 2. C드라이브 타겟 폴더 설정 및 생성
target_dir = "C:/Users/wpsol/wb/seoul/tta_test_data"
os.makedirs(target_dir, exist_ok=True)

new_filenames = []
new_filepaths = []
success_count = 0

print(f"▶ [{target_dir}] 폴더로 이미지 복사 및 이름 변경(TEST_N.jpg)을 시작합니다...")

# 3. 파일 복사 및 새로운 이름/경로 매핑
# 인덱스를 초기화하여 TEST_1부터 순서대로 번호가 부여되도록 합니다.
for idx, row in df_vision.reset_index(drop=True).iterrows():
    old_path = str(row['old_filepath'])
    lotid = row['LOTID']
    
    # 원본 파일에서 확장자만 추출 (누락 시 기본값 .jpg 부여)
    _, ext = os.path.splitext(old_path)
    if not ext: 
        ext = '.jpg'
    
    # 새로운 파일명(TEST_N) 및 C드라이브 경로 생성
    new_filename = f"TEST_{idx + 1}{ext}"
    new_filepath = os.path.join(target_dir, new_filename).replace("\\", "/")
    
    # D드라이브 원본 파일이 존재하는지 확인 후 C드라이브로 안전하게 복사
    if os.path.exists(old_path):
        try:
            shutil.copy2(old_path, new_filepath)
            success_count += 1
        except Exception as e:
            print(f"[오류] 복사 실패 ({old_path}): {e}")
            new_filepath = None
            new_filename = None
    else:
        new_filepath = None
        new_filename = None
        
    new_filenames.append(new_filename)
    new_filepaths.append(new_filepath)

# 매핑 결과를 데이터프레임에 저장
df_vision['FILENAME'] = new_filenames
df_vision['FILEPATH'] = new_filepaths
print(f"▶ 물리적 파일 복사 완료 (성공: {success_count}건 / 전체: {len(df_vision)}건)")

# 4. Target DB의 두 테이블에 FILENAME, FILEPATH 업데이트 (JOIN 방식)
print("▶ Target DB 테이블 경로 업데이트 및 스키마 복구 중...")

# 정상적으로 복사된 데이터만 남긴 매핑 테이블 생성
df_mapping = df_vision.dropna(subset=['FILEPATH'])[['LOTID', 'FILENAME', 'FILEPATH']]
tables_to_update = ['wm_raw_unstructed_datasets', 'wm_unstructed_datasets']

for table in tables_to_update:
    # Target DB에서 데이터 읽기 (이전에 추가했던 SEQ_ID도 함께 불러와짐)
    df_target = pd.read_sql(f"SELECT * FROM {table}", target_engine)
    
    # 기존에 경로 관련 컬럼이 있다면 충돌 방지를 위해 미리 제거
    cols_to_drop = [col for col in ['FILENAME', 'FILEPATH', 'FILE_PATH'] if col in df_target.columns]
    if cols_to_drop:
        df_target = df_target.drop(columns=cols_to_drop)
        
    # LOTID를 기준으로 새로운 파일명과 경로 매핑 (Left Join)
    df_merged = pd.merge(df_target, df_mapping, on='LOTID', how='left')
    
    # 다시 DB에 적재 및 기본키(PK) 제약조건 복구
    with target_engine.begin() as connection:
        df_merged.to_sql(name=table, con=target_engine, if_exists='replace', index=False)
        
        # LOTID 타입 지정 및 SEQ_ID 복합키/자동증가 속성 완벽 복구
        connection.execute(text(f"ALTER TABLE {table} MODIFY LOTID VARCHAR(50);"))
        connection.execute(text(f"ALTER TABLE {table} MODIFY SEQ_ID INT AUTO_INCREMENT PRIMARY KEY;"))
        
    print(f"  - {table} 업데이트 완료.")

print("\n▶ [최종 완료] Step 4, 5, 7 파이프라인 (이미지 복사, 이름 변경, DB 동기화)이 완료되었습니다!")
display(df_mapping.head())

In [ ]:
tables_to_update = ['wm_raw_unstructed_datasets', 'wm_unstructed_datasets']

print("▶ FILENAME 기반 LOTID 변환 작업(SWT-TEST_N)을 시작합니다...")

for table in tables_to_update:
    # 1. Target DB에서 기존 데이터 로드
    df_current = pd.read_sql(f"SELECT * FROM {table}", target_engine)
    
    if df_current.empty or 'FILENAME' not in df_current.columns:
        print(f"  - [경고] {table} 테이블이 비어있거나 FILENAME 컬럼이 없어 스킵합니다.")
        continue
        
    # 2. FILENAME 컬럼에서 숫자(\d+)만 추출 (예: TEST_1.jpg -> 1)
    # 결측치 방지를 위해 str 타입으로 변환 후 추출 진행
    df_current['extracted_num'] = df_current['FILENAME'].astype(str).str.extract(r'(\d+)')
    
    # 3. 새로운 양식(SWT-TEST_N)으로 LOTID 컬럼 갱신
    df_current['LOTID'] = 'SWT-TEST_' + df_current['extracted_num']
    
    # 임시 사용한 컬럼 제거
    df_current = df_current.drop(columns=['extracted_num'])
    
    # 4. 변경된 데이터프레임으로 DB 테이블 덮어쓰기
    with target_engine.begin() as connection:
        df_current.to_sql(
            name=table,
            con=connection,
            if_exists='replace',
            index=False
        )
        
        # 5. 무너진 DB 스키마 및 제약조건(PK) 복구
        connection.execute(text(f"ALTER TABLE {table} MODIFY LOTID VARCHAR(50);"))
        connection.execute(text(f"ALTER TABLE {table} MODIFY SEQ_ID INT AUTO_INCREMENT PRIMARY KEY;"))
        
    print(f"  - {table} 테이블 LOTID 변경 및 PK 복구 완료.")

print("\n▶ [최종 완료] 모든 테이블의 LOTID가 SWT-TEST_N 양식으로 정상 변경되었습니다.")

In [ ]:
tables_to_update = ['wm_unstructed_datasets']

print("▶ FILENAME 기준 데이터 정렬 및 SEQ_ID 재부여 작업을 시작합니다.")

for table in tables_to_update:
    # 1. Target DB에서 기존 데이터 로드
    df = pd.read_sql(f"SELECT * FROM {table}", target_engine)
    
    if df.empty or 'FILENAME' not in df.columns:
        print(f"  - [경고] {table}에 데이터가 없거나 FILENAME 컬럼이 없습니다.")
        continue
        
    # 2. FILENAME에서 숫자만 추출하여 정수(int)형 정렬 기준으로 사용
    df['sort_num'] = df['FILENAME'].astype(str).str.extract(r'(\d+)').astype(int)
    
    # 3. 숫자 기준으로 오름차순 정렬 후, 기존 SEQ_ID와 임시 정렬 컬럼 삭제
    df_sorted = df.sort_values(by='sort_num').reset_index(drop=True)
    df_sorted = df_sorted.drop(columns=['SEQ_ID', 'sort_num'], errors='ignore')
    
    # 4. 정렬된 데이터를 DB에 덮어쓰고 SEQ_ID를 새롭게 생성
    with target_engine.begin() as connection:
        df_sorted.to_sql(
            name=table,
            con=connection,
            if_exists='replace',
            index=False
        )
        
        # 테이블 맨 앞에 SEQ_ID 추가 (삽입된 정렬 순서대로 1부터 자동 부여됨)
        connection.execute(text(f"ALTER TABLE {table} MODIFY LOTID VARCHAR(50);"))
        connection.execute(text(f"ALTER TABLE {table} ADD COLUMN SEQ_ID INT AUTO_INCREMENT PRIMARY KEY FIRST;"))
        
    print(f"  - {table} 테이블 정렬 및 SEQ_ID 재부여 완료.")

print("\n▶ [최종 완료] 모든 테이블의 데이터가 FILENAME 기준으로 깔끔하게 정렬되었습니다.")

In [ ]:
tables_to_check = ['wm_raw_unstructed_datasets', 'wm_unstructed_datasets']

print("▶ 테이블별 DATIME(공정 발생 일시) 범위 확인을 시작합니다.\n")

for table in tables_to_check:
    # DB 레벨에서 최소/최대 날짜만 즉시 연산하여 가져옵니다.
    query = f"SELECT MIN(DATIME) AS min_date, MAX(DATIME) AS max_date FROM {table}"
    
    try:
        df_dates = pd.read_sql(query, target_engine)
        min_date = df_dates['min_date'].iloc[0]
        max_date = df_dates['max_date'].iloc[0]
        
        print(f"[{table}]")
        print(f"  - 최소 날짜(시작일) : {min_date}")
        print(f"  - 최대 날짜(종료일) : {max_date}\n")
    except Exception as e:
        print(f"[{table}] 조회 중 오류 발생: {e}\n")

In [ ]:
tables_to_update = ['wm_raw_unstructed_datasets', 'wm_unstructed_datasets']

# 1. 53일의 시간 차이 객체 생성
time_shift = pd.Timedelta(days=53)

print("▶ DATIME 일괄 평행 이동(+53일) 작업을 시작합니다...")

for table in tables_to_update:
    # DB에서 기존 데이터 로드
    df = pd.read_sql(f"SELECT * FROM {table}", target_engine)
    
    if df.empty or 'DATIME' not in df.columns:
        print(f"  - [경고] {table}에 데이터가 없거나 DATIME 컬럼이 없습니다.")
        continue
        
    # 2. DATIME 컬럼을 datetime 타입으로 변환 후 53일 더하기
    df['DATIME'] = pd.to_datetime(df['DATIME']) + time_shift
    
    # 3. DB 적재를 위해 다시 원래의 문자열 포맷(YYYY-MM-DD HH:MM:SS.000)으로 변경
    df['DATIME'] = df['DATIME'].dt.strftime('%Y-%m-%d %H:%M:%S.000')
    
    # 4. DB 덮어쓰기 및 제약조건 복구
    with target_engine.begin() as connection:
        df.to_sql(
            name=table,
            con=connection,
            if_exists='replace',
            index=False
        )
        
        # 이전 단계에서 유지된 SEQ_ID를 다시 PK로 설정
        connection.execute(text(f"ALTER TABLE {table} MODIFY LOTID VARCHAR(50);"))
        connection.execute(text(f"ALTER TABLE {table} MODIFY SEQ_ID INT AUTO_INCREMENT PRIMARY KEY;"))
        
    print(f"  - {table} 테이블 DATIME 갱신 및 스키마 복구 완료.")

print("\n▶ [최종 완료] 모든 데이터의 DATIME이 간격 훼손 없이 성공적으로 평행 이동되었습니다.")